In [1]:
# %pip install -q transformers datasets lm-eval accelerate

In [2]:
import os
os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ["HF_DATASETS_CACHE"] = "/root/.cache/huggingface/datasets"

In [3]:
from datasets import load_dataset
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer

In [27]:
import json
from tqdm import tqdm
import math, torch
import importlib.metadata as md

In [5]:
model_dir = "/workspace/language_adaptation/olmo2-7b-stage1-step928646-tokens3896B"

#### Loading tokenizer, dataset, and calculating tokenizer_stats, bits_per_byte

In [10]:
# Loading the tokenizer
tok = AutoTokenizer.from_pretrained(model_dir)

In [11]:
# Load kannada dataset
kn = load_dataset(
    "ai4bharat/sangraha",
    data_dir="verified/kan",
    split="train",
    cache_dir="/workspace/language_adaptation/dataset",
)

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

In [12]:
print(kn)                  # shows columns and row count
print(kn[0]["text"][:500])

Dataset({
    features: ['doc_id', 'text', 'type'],
    num_rows: 3632345
})
ನಗರದ ಕುಮಾರಸ್ವಾಮಿ ದೇವಸ್ಥಾನದ ಮುಖ್ಯರಸ್ತೆ ಬಳಿ ಇರುವ ಕುರುಬರ ಸಂಘದ ಆವರಣದಲ್ಲಿ ವಸತಿ ನಿಲಯ ಕಟ್ಟಡ ನಿರ್ಮಾಣಕ್ಕಾಗಿ ಅಗೆದಿದ್ದ ಅಂದಾಜು 10 ಅಡಿ ಆಳದ ಬುನಾದಿಯಲ್ಲಿ ಬಿದ್ದ ಹೋರಿಯನ್ನು ಅಗ್ನಿಶಾಮಕದಳ ಸಿಬ್ಬಂದಿ ಭಾನುವಾರ ರಕ್ಷಿಸಿದರು.
 ಬಳ್ಳಾರಿ : ನಗರದ ಕುಮಾರಸ್ವಾಮಿ ದೇವಸ್ಥಾನದ ಮುಖ್ಯರಸ್ತೆ ಬಳಿ ಇರುವ ಕುರುಬರ ಸಂಘದ ಆವರಣದಲ್ಲಿ ವಸತಿ ನಿಲಯ ಕಟ್ಟಡ ನಿರ್ಮಾಣಕ್ಕಾಗಿ ಅಗೆದಿದ್ದ ಅಂದಾಜು 10 ಅಡಿ ಆಳದ ಬುನಾದಿಯಲ್ಲಿ ಬಿದ್ದ ಹೋರಿಯನ್ನು ಅಗ್ನಿಶಾಮಕದಳ ಸಿಬ್ಬಂದಿ ಭಾನುವಾರ ರಕ್ಷಿಸಿದರು.
 ವಸತಿ ನಿಲಯ ಹಾಗೂ ವಾಣಿಜ್ಯ ಸಂಕೀರ್ಣ ಕಟ್ಟಡ ನಿರ್ಮಿಸಲು ಆಳ ಮತ್ತು ಅಗಲವಾದ ಬುನಾದಿಯನ್ನು ತೋಡಲಾಗಿದೆ. ಮೇವು ಅರಸಿ ಬಂ


In [13]:
# Disable caching as it leads to permission errors on /workspace directories
datasets.disable_caching()

In [14]:
# split = kn.train_test_split(test_size=2000, seed=42)
# kn_train, kn_eval = split["train"], split["test"]

In [15]:
# kn_texts = [t[:8000] for t in tqdm(kn_eval["text"]) if len(t) > 200]
# print(len(kn_texts))

In [16]:
# os.makedirs("/workspace/language_adaptation/eval", exist_ok=True)

# with open("/workspace/language_adaptation/eval/kn_eval_texts.jsonl", "w", encoding="utf-8") as f:
#     for t in kn_texts:
#         f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")

In [17]:
kn_texts = []

with open("/workspace/language_adaptation/eval/kn_eval_texts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        kn_texts.append(data["text"])

print(f"Successfully read {len(kn_texts)} lines.")   # Should be 1997

Successfully read 1997 lines.


In [18]:
def tokenizer_stats(tok, texts):
    n_tok = n_words = n_chars = n_frag = 0
    for t in tqdm(texts):
        ids = tok(t, add_special_tokens=False)["input_ids"]
        n_tok += len(ids)
        n_words += len(t.split())
        n_chars += len(t)
        n_frag += sum("\ufffd" in tok.decode([i]) for i in ids)
    return {
        "fertility (tokens/word)": n_tok / n_words,
        "chars/token": n_chars / n_tok,
        "fragment rate": n_frag / n_tok,
    }

In [19]:
tokenizer_stats(tok, kn_texts)

100%|██████████| 1997/1997 [00:14<00:00, 133.80it/s]


{'fertility (tokens/word)': 13.57003142523685,
 'chars/token': 0.595734263917983,
 'fragment rate': 0.9775849704648346}

In [31]:
tok_stats = {'fertility (tokens/word)': 13.57003142523685,
             'chars/token': 0.595734263917983,
             'fragment rate': 0.9775849704648346}

print(tok_stats)

{'fertility (tokens/word)': 13.57003142523685, 'chars/token': 0.595734263917983, 'fragment rate': 0.9775849704648346}


In [25]:
EVAL_DIR = "/workspace/language_adaptation/eval"

def save_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)  # atomic: never leaves a half-written file


In [28]:
eval_config = {
    "model": "allenai/OLMo-2-1124-7B", "revision": "stage1-step928646-tokens3896B", "dtype": "bf16",
    "tokenizer": "original", 
    "kn_eval": "/workspace/language_adaptation/eval/kn_eval_texts.jsonl",
    "bpb_max_len": 4096,
    "lm_eval": {"num_fewshot": 0, "limit": None},
    "versions": {p: md.version(p) for p in ["torch", "transformers", "datasets", "lm_eval"]},
}

In [29]:
lang, texts, tag = "kn", kn_texts, "baseline"   # tag = which tokenizer/model variant

In [32]:
save_json({"config": eval_config, "lang": lang, "n_docs": len(texts),
           "tokenizer_stats": tok_stats}, f"{EVAL_DIR}/{tag}_{lang}_tokenizer.json")

print("Tokenizer stats saved")

Tokenizer stats saved


In [20]:
model = AutoModelForCausalLM.from_pretrained(model_dir, dtype="bfloat16", device_map="cuda")

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

In [21]:
@torch.no_grad()
def bits_per_byte(model, tok, texts, max_len=4096, slice_len=1024):
    total_nll, total_bytes = 0.0, 0
    for t in tqdm(texts):
        ids = tok(t, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
        for s in range(0, len(ids) - 1, max_len):
            chunk = ids[s : s + max_len + 1].unsqueeze(0).to(model.device)
            if chunk.shape[1] < 2:
                continue
            inp, tgt = chunk[:, :-1], chunk[:, 1:]
            hidden = model.model(inp).last_hidden_state          # (1, L, d)
            for j in range(0, inp.shape[1], slice_len):
                logits = model.lm_head(hidden[:, j : j + slice_len]).float()
                total_nll += torch.nn.functional.cross_entropy(
                    logits[0], tgt[0, j : j + slice_len], reduction="sum"
                ).item()
        total_bytes += len(t.encode("utf-8"))
    return total_nll / (math.log(2) * total_bytes)

In [22]:
bits_per_byte(model, tok, kn_texts)

100%|██████████| 1997/1997 [42:15<00:00,  1.27s/it] 


0.847692143873666

In [24]:
bpb = 0.847692143873666

In [30]:
save_json({"config": eval_config, "lang": lang, "n_docs": len(texts),
           "bits_per_byte": bpb}, f"{EVAL_DIR}/{tag}_{lang}_bpb.json")
print(f"{lang} BPB: {bpb:.4f}")

kn BPB: 0.8477


In [ ]:
# torch.cuda.empty_cache()